# Multi-output model experimentation

This notebook runs the same validation, temporal feature engineering, candidate comparison, and bounded tuning pipeline used by the API. Observations at time *t* forecast the next observation. The bundled one-day dataset is for demonstration only; reliable performance needs a much larger and more geographically representative dataset.

In [ ]:
from pathlib import Path
import sys
import tempfile
import pandas as pd
import matplotlib.pyplot as plt

root_candidates = [Path.cwd(), Path.cwd().parent]
project_root = next(root for root in root_candidates if (root / "backend/app/ml/pipeline.py").exists())
sys.path.insert(0, str(project_root / "backend"))
from app.ml.pipeline import train_models

weather = pd.read_csv(project_root / "backend/datasets/sample_weather.csv")
result = train_models(weather, Path(tempfile.mkdtemp(prefix="weather-experiment-")))
pd.DataFrame(result["comparison"]).sort_values("r2", ascending=False)

In [ ]:
print(f"Selected: {result['model_name']} · {result['version']}")
print(f"Held-out rows: {len(result['actual_vs_predicted']) // 4}")
pd.DataFrame(result["feature_importance"]).head(12)

In [ ]:
temperature = pd.DataFrame(result["actual_vs_predicted"])
temperature = temperature[temperature["target"] == "temperature"].reset_index(drop=True)
residuals = pd.DataFrame(result["residuals"])
residuals = residuals[residuals["target"] == "temperature"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(temperature["actual"].to_numpy(), label="Actual")
axes[0].plot(temperature["predicted"].to_numpy(), label="Predicted")
axes[0].set_title("Temperature · chronological holdout")
axes[0].legend()
axes[1].scatter(residuals["actual"], residuals["residual"], alpha=0.75)
axes[1].axhline(0, color="black", linestyle="--", linewidth=1)
axes[1].set(title="Residuals vs actual", xlabel="Actual temperature", ylabel="Residual")
fig.tight_layout()